In [1]:
# Combines Blocks 1, 2, 4 into a single weekly excess return panel for MST
# estimation. Reads each block's Output/ CSV (already produced by that
# block's own notebook) and combines them into the N=130 test asset universe.
#
# Inputs (all in Output/, produced by the corresponding block notebook):
#   commodity_futures_weekly.csv                   -Block 1: 12 portfolios (commodity_futures_portfolios.ipynb)
#   commodity_equity_weekly_by_characteristics.csv -Block 2: 115 portfolios (commodity_equity_portfolios_by_characteristics.ipynb)
#   commodity_currency_weekly.csv                  -Block 4: 3 portfolios (commodity_currency_portfolios.ipynb)
#
# Outputs (both to Output/):
#   test_asset_universe.csv-N=130 x T panel, weekly Wednesday excess returns
#   pass3_factors.csv       -the 4 observable Pass 3 factors, restricted to the same date range

import pandas as pd

STUDY_START = '1996-02-07'   # driven by currency data availability
STUDY_END   = '2023-12-31'

# Load each block
b1 = pd.read_csv('Output/commodity_futures_weekly.csv', index_col=0, parse_dates=True)

b2 = pd.read_csv('Output/commodity_equity_weekly_by_characteristics.csv', index_col=0, parse_dates=True)

b4 = pd.read_csv('Output/commodity_currency_weekly.csv', index_col=0, parse_dates=True)

# Drop long-short factors from Block 1 and Block 4-keep tercile portfolios only
b1 = b1[[c for c in b1.columns if not c.startswith('r')]]   # drop rBasis, rMOM etc.
b4 = b4[[c for c in b4.columns if not c.startswith('r')]]   # drop rCURR

print(f"Block 1 (Futures)   : {b1.shape[1]:>3} portfolios  "
      f"{b1.index.min().date()} - {b1.index.max().date()}")
print(f"Block 2 (Equities)  : {b2.shape[1]:>3} portfolios  "
      f"{b2.index.min().date()} - {b2.index.max().date()}")
print(f"Block 4 (Currencies): {b4.shape[1]:>3} portfolios  "
      f"{b4.index.min().date()} - {b4.index.max().date()}")

# Prefix columns with block label
b1.columns = ['B1_' + c for c in b1.columns]
b2.columns = ['B2_' + c for c in b2.columns]
b4.columns = ['B4_' + c for c in b4.columns]

# Align to common Wednesday index via outer join, then restrict
universe = (b1.join(b2, how='outer')
              .join(b4, how='outer')
              .sort_index()
              .loc[STUDY_START:STUDY_END])

# Coverage diagnostics
print(f"\nCombined universe:")
print(f"  Shape   : {universe.shape[0]:,} weeks x {universe.shape[1]} portfolios")
print(f"  Period  : {universe.index.min().date()} - {universe.index.max().date()}")

coverage = universe.notna().mean()
print(f"\nCoverage by block:")
for prefix, label in [('B1_', 'Block 1'), ('B2_', 'Block 2'), ('B4_', 'Block 4')]:
    cols = [c for c in universe.columns if c.startswith(prefix)]
    cov  = coverage[cols].mean()
    low  = coverage[cols].min()
    print(f"  {label}: {len(cols):>3} cols  mean={cov:.1%}  min={low:.1%}")

print(f"\nWeeks with ≥ 90% coverage: "
      f"{(universe.notna().mean(axis=1) >= 0.9).sum():,}")
print(f"Weeks with any data       : "
      f"{universe.notna().any(axis=1).sum():,}")

# Missing pattern
print(f"\nFirst fully-observed week (all blocks): ", end='')
full = universe.notna().all(axis=1)
print(full[full].index.min().date() if full.any() else 'none')

# Save
out_path = 'Output/test_asset_universe.csv'
universe.to_csv(out_path)
print(f"\nSaved: test_asset_universe.csv  {universe.shape}")

# Also save the 4 observable factors for Pass 3
factors = pd.read_csv('Output/commodity_factors_weekly.csv', index_col=0, parse_dates=True)
factors = factors.loc[STUDY_START:STUDY_END]
factors.to_csv('Output/pass3_factors.csv')
print(f"Saved: pass3_factors.csv {factors.shape}")


Block 1 (Futures)   :  12 portfolios  1990-01-03 - 2023-12-27
Block 2 (Equities)  : 115 portfolios  1995-07-05 - 2023-12-27
Block 4 (Currencies):   3 portfolios  1996-02-07 - 2023-12-27

Combined universe:
  Shape   : 1,456 weeks x 130 portfolios
  Period  : 1996-02-07 - 2023-12-27

Coverage by block:
  Block 1:  12 cols  mean=99.6%  min=98.5%
  Block 2: 115 cols  mean=100.0%  min=100.0%
  Block 4:   3 cols  mean=100.0%  min=100.0%

Weeks with ≥ 90% coverage: 1,456
Weeks with any data       : 1,456

First fully-observed week (all blocks): 1996-02-07

Saved: test_asset_universe.csv  (1456, 130)
Saved: pass3_factors.csv (1456, 4)
